In [4]:
!pip install datasets

  Using cached datasets-4.8.2-py3-none-any.whl.metadata (19 kB)
  Using cached pyarrow-23.0.1-cp310-cp310-win_amd64.whl.metadata (3.1 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.6.0-cp310-cp310-win_amd64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.19-py310-none-any.whl.metadata (7.5 kB)
  Using cached huggingface_hub-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached aiohttp-3.13.3-cp310-cp310-win_amd64.whl.metadata (8.4 kB)
  Using cached hf_xet-1.4.2-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached frozenlist-1.8.0-cp310-cp310-win_amd64.whl.metadata (21 kB)
  Using cached multidict-6.7.1-cp310-cp310-win_amd64.whl.metadata (5.5 kB)
  Using cached propcac


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from datasets import load_dataset

In [2]:
dataset = load_dataset("amazon_polarity")

train_data = dataset["train"].select(range(20000))
test_data = dataset["test"].select(range(5000))

In [7]:
train_data[0]

{'label': 1,
 'title': 'Stuning even for the non-gamer',
 'content': 'This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^'}

In [6]:
def tokenize(text):
    return text.lower().split()

In [42]:
from collections import Counter
def build_vocab(texts, min_freq=2):
    counter = Counter()

    for text in texts:
        counter.update(tokenize(text))

    vocab = {"<PAD>": 0, "<UNK>": 1}

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

vocab = build_vocab([sample["content"] for sample in train_data], min_freq=2)

In [8]:
from collections import Counter

def build_vocab(texts, min_freq=2):
    counter = Counter()

    for text in texts:
        counter.update(tokenize(text))

    vocab = {"<pad>": 0, "<unk>": 1}

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

In [38]:
vocab = build_vocab([sample["content"] for sample in train_data], min_freq=2)
vocab_size = len(vocab)
vocab_size

35366

In [23]:
vocab_size

35366

In [43]:
max_len = 100

def encode(text):
    tokens = tokenize(text)
    seq = [vocab.get(word, 1) for word in tokens]
    return seq[:max_len]

def pad(seq):
    return seq + [0]*(max_len - len(seq))

In [40]:
def encode(text, vocab, max_len = 100):
    return [vocab.get(word, vocab["<unk>"]) for word in tokenize(text)][:max_len]

def pad(sequences, max_len):
    padded = []

    for seq in sequences:
        seq = seq[:max_len]
        seq = seq + [0] * (max_len - len(seq))  # pad=0
        padded.append(seq)

    return padded

In [45]:
class AmazonDataset(torch.utils.data.Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        text = self.data[idx]["content"]
        label = self.data[idx]["label"]

        seq = pad(encode(text))

        return torch.tensor(seq), torch.tensor(label)

In [46]:
train_loader = DataLoader(
    AmazonDataset(train_data),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    AmazonDataset(test_data),
    batch_size=64
)

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        # x: (batch, seq_len)

        x = self.embedding(x)
        # (batch, seq_len, embed_dim)

        output, hidden = self.rnn(x)
        # hidden: (1, batch, hidden_dim)

        return self.fc(hidden.squeeze(0))

In [47]:
class BetterRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.rnn = nn.RNN(
            embed_dim,
            hidden_dim,
            num_layers=2,
            nonlinearity='relu',
            batch_first=True
        )

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.embedding(x)

        output, hidden = self.rnn(x)

        out = torch.mean(output, dim=1)

        out = self.dropout(out)

        return self.fc(out)

In [56]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
simple_model = SimpleRNN(
    vocab_size= vocab_size,
    embed_dim=100,
    hidden_dim=128
).to(device)
model = BetterRNN(
    vocab_size= vocab_size,
    embed_dim=100,
    hidden_dim=128
).to(device)

Using device: cuda


In [49]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [58]:
epochs = 20

for epoch in range(epochs):

    for text, label in train_loader:

        text = text.to(device)
        label = label.to(device)

        outputs = simple_model(text)

        loss = criterion(outputs, label)

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(simple_model.parameters(), 5)

        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6948
Epoch 2, Loss: 0.7455
Epoch 3, Loss: 0.6796
Epoch 4, Loss: 0.7045
Epoch 5, Loss: 0.7389
Epoch 6, Loss: 0.7372
Epoch 7, Loss: 0.7283
Epoch 8, Loss: 0.6991
Epoch 9, Loss: 0.7108
Epoch 10, Loss: 0.6883
Epoch 11, Loss: 0.7325
Epoch 12, Loss: 0.6605
Epoch 13, Loss: 0.7023
Epoch 14, Loss: 0.7097
Epoch 15, Loss: 0.7034
Epoch 16, Loss: 0.7219
Epoch 17, Loss: 0.6982
Epoch 18, Loss: 0.6829
Epoch 19, Loss: 0.6691
Epoch 20, Loss: 0.7162


In [59]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for texts, labels in test_loader:

        texts = texts.to(device)
        labels = labels.to(device)

        outputs = simple_model(texts)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 48.46


In [ ]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for texts, labels in test_loader:

        texts = texts.to(device)
        labels = labels.to(device)

        outputs = model(texts) #better rnn model

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 80.78


In [52]:
def predict(text):

    model.eval()

    with torch.no_grad():

        seq = pad(encode(text))
        seq = torch.tensor(seq).unsqueeze(0).to(device)

        output = model(seq)

        _, pred = torch.max(output,1)

        return "Positive" if pred.item() == 1 else "Negative"

In [53]:
print(predict("this product is amazing"))
print(predict("don't waste your money on this"))

Positive
Negative


LSTM

In [83]:
class LSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=2,
            batch_first=True,
            # bidirectional=True  
        )

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(hidden_dim, 2) 

    def forward(self, x):
        x = self.embedding(x)
        # (batch, seq_len, embed_dim)

        output, (hidden, cell) = self.lstm(x)
        # hidden: (num_layers*2, batch, hidden_dim)

        out = torch.mean(output, dim=1)   # or torch.max
        out = self.dropout(out)
        return self.fc(out)

       

In [84]:
lstm_model = LSTM(
    vocab_size= vocab_size,
    embed_dim=200,
    hidden_dim=128
).to(device)
print(lstm_model)

LSTM(
  (embedding): Embedding(35366, 200)
  (lstm): LSTM(200, 128, num_layers=2, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


In [95]:
train_loader.dataset.data[10]

{'label': 0,
 'title': 'The Worst!',
 'content': "A complete waste of time. Typographical errors, poor grammar, and a totally pathetic plot add up to absolutely nothing. I'm embarrassed for this author and very disappointed I actually paid for this book."}

In [ ]:
for epoch in range(10):
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        xb = xb.to(device)
        yb = yb.to(device)

        output = lstm_model(xb)
        loss = criterion(output, yb)

        loss.backward()

        # gradient clipping (still useful)
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), 5)

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [80]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for texts, labels in test_loader:

        texts = texts.to(device)
        labels = labels.to(device)

        outputs = lstm_model(texts)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 51.86
